# Training Uncertainty Quantification Models

This notebook trains five uncertainty quantification methods and exports them for analysis.

**Run this notebook once** to generate trained models, then use `analyze_posterior.ipynb` for the polished analysis and visualization.

## Models Trained

| Method | Type | Training Time |
|--------|------|---------------|
| **FNN** | Deterministic | ~10s |
| **DropoutFNN** | MC Dropout | ~10s |
| **BayesianFNN** | Variational Inference | ~30s |
| **LaplaceFNN** | Post-hoc (on FNN) | ~1s |
| **MCMCFNN** | Tempered Posterior (NUTS) | ~2-5min |

## Output

All models are saved to `models/` directory with the following structure:
```
models/
├── fnn/           # Orbax checkpoint + metadata
├── dropoutfnn/
├── bayesianfnn/
├── laplacefnn/    # Posterior + MAP params
└── mcmcfnn/       # Posterior samples
```


In [ ]:
from pathlib import Path

import jax
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from bayescal.models import FNN, MCMCFNN, BayesianFNN, DropoutFNN, LaplaceFNN
from bayescal.utils import ModelResult
from bayescal.utils.model_export import export_model, export_laplace, export_mcmc
from bayescal.utils.toy_dataset import generate_concentric_rings_dataset
from bayescal.utils.toy_training import train_model

# Create model directory for saving exported models
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
print(f"Models will be saved to: {MODELS_DIR.absolute()}")

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)
print("Setup complete!")


Models will be saved to: /Users/kamilkrukowski/Code/BayesNNBench/notebooks/models
Setup complete!


## 1. Generate Dataset

Concentric rings with sinusoidal probability field: $P(y=1 | \mathbf{x}) = \sigma(A \cdot \sin(\omega \cdot r))$


In [21]:
# Dataset configuration
X_RANGE = (-5.0, 5.0)
Y_RANGE = (-5.0, 5.0)

# Generate dataset
X, y, y_onehot, prob_func = generate_concentric_rings_dataset(
    n_samples=3000,
    x_range=X_RANGE,
    y_range=Y_RANGE,
    min_prob=0.05,
    max_prob=0.95,
    seed=SEED,
)

# Split into train/test (10% train, 90% test for calibration analysis)
X_train, X_test, y_train, y_test, y_train_onehot, y_test_onehot = train_test_split(
    X, y, y_onehot, test_size=0.9, random_state=SEED, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")


Loaded dataset from cache: .cache/datasets/concentric_rings_3c07815303fd.npz
Training set: 300 samples
Test set: 2700 samples


## 2. Train Models

All models share the same architecture: **3 hidden layers × 32 units**.

### 2.1 Training-Time Methods


In [22]:
# Shared architecture
HIDDEN_DIMS = (32, 32, 32)
NUM_CLASSES = 2

# ============================================================================
# FNN (Deterministic Baseline)
# ============================================================================
print("=" * 60)
print("Training FNN (deterministic baseline)...")
print("=" * 60)

fnn_model = FNN(hidden_dims=HIDDEN_DIMS, num_classes=NUM_CLASSES, dropout_rate=0.0)
fnn_params, fnn_history = train_model(
    fnn_model, "FNN", X_train, y_train_onehot, epochs=200, seed=SEED
)

export_model(fnn_model, fnn_params, input_shape=(2,), output_path=MODELS_DIR, model_name="FNN", seed=SEED)
print()


Training FNN (deterministic baseline)...
FNN - Epoch 50/200: Loss=0.5404, Acc=0.7395
FNN - Epoch 100/200: Loss=0.4103, Acc=0.7881
FNN - Epoch 150/200: Loss=0.3533, Acc=0.8205
FNN - Epoch 200/200: Loss=0.3347, Acc=0.8281



In [23]:
# ============================================================================
# DropoutFNN (MC Dropout)
# ============================================================================
print("=" * 60)
print("Training DropoutFNN (MC Dropout)...")
print("=" * 60)

dropout_model = DropoutFNN(hidden_dims=HIDDEN_DIMS, num_classes=NUM_CLASSES, dropout_rate=0.2)
dropout_params, dropout_history = train_model(
    dropout_model, "DropoutFNN", X_train, y_train_onehot, epochs=300, seed=SEED
)

export_model(dropout_model, dropout_params, input_shape=(2,), output_path=MODELS_DIR, model_name="DropoutFNN", seed=SEED)
print()


Training DropoutFNN (MC Dropout)...
DropoutFNN - Epoch 50/300: Loss=0.6647, Acc=0.5526
DropoutFNN - Epoch 100/300: Loss=0.6222, Acc=0.6472
DropoutFNN - Epoch 150/300: Loss=0.5885, Acc=0.6787
DropoutFNN - Epoch 200/300: Loss=0.5838, Acc=0.6787
DropoutFNN - Epoch 250/300: Loss=0.5878, Acc=0.7227
DropoutFNN - Epoch 300/300: Loss=0.5897, Acc=0.6798



In [24]:
# ============================================================================
# BayesianFNN (Variational Inference / Bayes by Backprop)
# ============================================================================
print("=" * 60)
print("Training BayesianFNN (Variational Inference)...")
print("=" * 60)

bayesian_model = BayesianFNN(
    hidden_dims=HIDDEN_DIMS,
    num_classes=NUM_CLASSES,
    beta=0.05,              # KL penalty weight
    posterior_std_init=0.1,
    max_std=0.1,            # Cap sigma to prevent excessive noise
)
bayesian_params, bayesian_history = train_model(
    bayesian_model,
    "BayesianFNN",
    X_train,
    y_train_onehot,
    epochs=400,
    lr=0.0015,
    seed=SEED,
    warm_up_epochs=50,
)

export_model(bayesian_model, bayesian_params, input_shape=(2,), output_path=MODELS_DIR, model_name="BayesianFNN", seed=SEED)
print()


Training BayesianFNN (Variational Inference)...
BayesianFNN - Epoch 50/400: Loss=1.3410, Acc=0.5097, Likelihood=0.6837, KL=3944, KL_norm=13.1463, β*KL_norm=0.6442 (warmup β=0.049000), σ_mean=0.1000, σ_median=0.1000, σ_max=0.1000
BayesianFNN - Epoch 100/400: Loss=1.3256, Acc=0.5687, Likelihood=0.6683, KL=3944, KL_norm=13.1469, β*KL_norm=0.6573, σ_mean=0.1000, σ_median=0.1000, σ_max=0.1000
BayesianFNN - Epoch 150/400: Loss=1.3218, Acc=0.5994, Likelihood=0.6644, KL=3944, KL_norm=13.1476, β*KL_norm=0.6574, σ_mean=0.0999, σ_median=0.1000, σ_max=0.1000
BayesianFNN - Epoch 200/400: Loss=1.3287, Acc=0.5926, Likelihood=0.6711, KL=3945, KL_norm=13.1515, β*KL_norm=0.6576, σ_mean=0.0999, σ_median=0.1000, σ_max=0.1000
BayesianFNN - Epoch 250/400: Loss=1.3119, Acc=0.6241, Likelihood=0.6542, KL=3946, KL_norm=13.1547, β*KL_norm=0.6577, σ_mean=0.0999, σ_median=0.1000, σ_max=0.1000
BayesianFNN - Epoch 300/400: Loss=1.3056, Acc=0.6054, Likelihood=0.6474, KL=3949, KL_norm=13.1646, β*KL_norm=0.6582, σ_mean

### 2.2 Post-hoc Methods

These methods add uncertainty quantification to a pre-trained model (FNN).


In [25]:
# ============================================================================
# LaplaceFNN (Post-hoc Laplace Approximation on trained FNN)
# ============================================================================
print("=" * 60)
print("Fitting LaplaceFNN (post-hoc Laplace approximation)...")
print("=" * 60)

laplace_model = LaplaceFNN.fit(
    base_model=fnn_model,
    params=fnn_params,
    X_train=X_train,
    y_train=y_train_onehot,
    prior_precision=10.0,
    subset_size=1000,
)

export_laplace(laplace_model, output_path=MODELS_DIR, model_name="LaplaceFNN")
print()


Fitting LaplaceFNN (post-hoc Laplace approximation)...
Exported LaplaceFNN to models/laplacefnn



In [ ]:
# ============================================================================
# MCMCFNN (NUTS Sampling - Tempered Posterior via MCMC)
# ============================================================================
print("=" * 60)
print("Fitting MCMCFNN (NUTS sampling with tempered posterior)...")
print("Note: Using temperature=0.05 (cold posterior)")
print("      This is NOT the true Bayesian posterior, but a tempered posterior")
print("      approximated via MCMC (NUTS). MCMC yields approximate posterior")
print("      expectations with Monte Carlo error.")
print("      This is the slowest method (~2-5 minutes)")
print("=" * 60)

mcmc_model = MCMCFNN.fit(
    hidden_dims=HIDDEN_DIMS,
    num_classes=NUM_CLASSES,
    X_train=X_train,
    y_train=y_train_onehot,
    prior_std=0.1,
    temperature=0.05,   # Cold posterior (tempered, not true Bayesian posterior)
    num_warmup=50,
    num_samples=50,
    sampler="nuts",
    seed=SEED,
)

export_mcmc(mcmc_model, output_path=MODELS_DIR, model_name="MCMCFNN")
print()


Fitting MCMCFNN (NUTS sampling)...
Note: This is the slowest method (~2-5 minutes)
MCMCFNN: 2274 parameters to sample
Using cold posterior with temperature=0.05
Running NUTS sampling...
Sampling complete. Shape: (50, 2274)
Exported MCMCFNN to models/mcmcfnn



## 3. Test Set Evaluation

Quick evaluation on the held-out test set (90% of data).


In [27]:
# Evaluate all models on test set
NUM_CAL_BINS = 50
MC_SAMPLES = 100

all_results = [
    ModelResult("FNN", fnn_model, fnn_params, n_samples=1, color="#F18F01"),
    ModelResult("DropoutFNN", dropout_model, dropout_params, n_samples=MC_SAMPLES, color="#A23B72"),
    ModelResult("BayesianFNN", bayesian_model, bayesian_params, n_samples=MC_SAMPLES, color="#2E86AB"),
    ModelResult("LaplaceFNN", laplace_model, {}, n_samples=MC_SAMPLES, color="#3D5A80"),
    ModelResult("MCMCFNN", mcmc_model, {}, n_samples=50, color="#6B9080"),
]

# Evaluate
for result in all_results:
    result.evaluate(X_test, y_test_onehot, seed=SEED, num_bins=NUM_CAL_BINS)

# Display results table
results_df = pd.DataFrame([r.to_metrics_row() for r in all_results])
results_df = results_df.rename(columns={
    "accuracy": "Accuracy", "ece": "ECE", "mce": "MCE", "brier": "Brier"
})
results_df = results_df[["Model", "Accuracy", "ECE", "MCE", "Brier"]]

print("\n" + "=" * 60)
print("TEST SET RESULTS")
print("=" * 60)
print(results_df.to_string(index=False, float_format="%.4f"))
print("=" * 60)
print(f"\nBest ECE: {results_df.loc[results_df['ECE'].idxmin(), 'Model']}")
print(f"Best Accuracy: {results_df.loc[results_df['Accuracy'].idxmax(), 'Model']}")



TEST SET RESULTS
      Model  Accuracy    ECE    MCE  Brier
        FNN    0.6793 0.1655 0.2632 0.4682
 DropoutFNN    0.6896 0.0949 0.2500 0.4279
BayesianFNN    0.7385 0.1072 0.1886 0.3913
 LaplaceFNN    0.6659 0.0716 0.1845 0.4228
    MCMCFNN    0.7130 0.0798 0.2160 0.4088

Best ECE: LaplaceFNN
Best Accuracy: BayesianFNN


## 4. Training Complete

All models have been trained, evaluated, and exported to `models/`. 

For polished visualizations (calibration curves, posterior plots, uncertainty maps), run `analyze_posterior.ipynb`.


In [28]:
# List exported models
print("Exported models:")
print("-" * 40)
for model_dir in sorted(MODELS_DIR.iterdir()):
    if model_dir.is_dir():
        files = list(model_dir.rglob("*"))
        size_kb = sum(f.stat().st_size for f in files if f.is_file()) / 1024
        print(f"  {model_dir.name:15} ({size_kb:.1f} KB)")


Exported models:
----------------------------------------
  bayesianfnn     (21.1 KB)
  dropoutfnn      (16.3 KB)
  fnn             (17.5 KB)
  laplacefnn      (29.4 KB)
  mcmcfnn         (445.4 KB)
